In [1]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv("../.env")

# 환경 변수에서 API 키 가져오기
#os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") #type:ignore


True

In [5]:

# 쿼리를 위한 로그 설정
import logging

logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)


In [3]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 로더 설정
loaders = [TextLoader("Data/How_to_invest_money.txt")]

docs = []
for loader in loaders:
    docs.extend(loader.load())

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_16752\209222700.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
# 문서 생성을 위한 텍스트 분할기 정의
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# 문서 분할
split_docs = recursive_splitter.split_documents(docs)

# OpenAIEmbeddings 인스턴스 생성
embeddings = OpenAIEmbeddings()

# Chroma vectorstore 생성
vectorstore = Chroma.from_documents(documents=split_docs, embedding=embeddings)

In [6]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# LLM 모델 설정 (여기서는 ChatOpenAI 사용)
llm = ChatOpenAI(model = "gpt-4o",temperature=0.2)

# MultiQueryRetriever 실행
retriever = MultiQueryRetriever.from_llm(
retriever=vectorstore.as_retriever(), # 기본 검색기 (벡터 데이터베이스)
llm=llm, # 앞서 정의한 llm (gpt-4o)
)

In [7]:
# 샘플 질문
question = "주식 투자를 처음 시작하려면 어떻게 해야 하나요?"

# 결과 검색
unique_docs = retriever.invoke(question)
print(f"\n결과: {len(unique_docs)}개의 문서가 검색되었습니다.")

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. 주식 투자 초보자가 알아야 할 기본 단계는 무엇인가요?', '2. 주식 투자를 처음으로 시작할 때 필요한 준비 사항은 무엇인가요?', '3. 주식 시장에 처음 진입할 때 어떤 절차를 따라야 하나요?']



결과: 7개의 문서가 검색되었습니다.


In [8]:
from langchain_classic.chains import RetrievalQA

# RetrievalQA 체인 설정
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# 질문에 대한 답변 생성
result = qa_chain.invoke({"query": question})

# 결과 출력
print("답변:", result["result"])
print("\n사용된 문서:")
for doc in result["source_documents"]:
    print(doc.page_content)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. 주식 투자 초보자가 알아야 할 기본 단계는 무엇인가요?', '2. 처음으로 주식 투자를 시작할 때 필요한 준비 사항은 무엇인가요?', '3. 주식 투자 입문자를 위한 가이드라인이나 조언이 있나요?']


답변: 주식 투자를 처음 시작하려면 다음과 같은 단계를 고려할 수 있습니다:

1. **기본 개념 이해하기**: 주식과 채권의 차이점, 주식 시장의 작동 방식, 주식의 가치 평가 방법 등을 이해하는 것이 중요합니다. 이는 투자 결정을 내리는 데 필요한 기초 지식을 제공합니다.

2. **목표 설정하기**: 투자 목표를 명확히 설정하세요. 예를 들어, 장기적인 자산 증식을 목표로 할 것인지, 단기적인 수익을 추구할 것인지 결정해야 합니다.

3. **리스크 분산**: 다양한 주식에 투자하여 리스크를 분산시키는 것이 중요합니다. 이는 특정 주식의 가격 변동에 대한 노출을 줄이는 데 도움이 됩니다.

4. **시장 조사 및 분석**: 투자하려는 회사의 재무 상태, 시장 위치, 성장 가능성 등을 조사하고 분석하세요. 이는 더 나은 투자 결정을 내리는 데 도움이 됩니다.

5. **투자 계좌 개설**: 주식 거래를 위해 증권사에서 투자 계좌를 개설해야 합니다. 여러 증권사의 수수료와 서비스 수준을 비교하여 자신에게 맞는 곳을 선택하세요.

6. **전문가 상담**: 투자에 대한 경험이 부족하다면, 신뢰할 수 있는 투자 전문가나 금융 컨설턴트와 상담하는 것도 좋은 방법입니다. 이는 더 나은 투자 전략을 수립하는 데 도움이 될 수 있습니다.

7. **작은 금액으로 시작하기**: 처음에는 작은 금액으로 시작하여 경험을 쌓고, 점차 투자 금액을 늘려가는 것이 좋습니다.

8. **지속적인 학습**: 금융 시장은 끊임없이 변화하므로, 지속적으로 학습하고 시장 동향을 파악하는 것이 중요합니다.

이러한 단계를 통해 주식 투자에 대한 기초를 다지고, 더 나은 투자 결정을 내릴 수 있을 것입니다.

사용된 문서:
G. G. H.




HOW TO INVEST MONEY




I

GENERAL PRINCIPLES OF INVESTMENT
For the successful investment of money, however, a good deal more is
required t